# Deploy MCP Servers to AWS Bedrock AgentCore Gateway

## 🎯 Workshop Overview

Welcome to this hands-on workshop! You'll learn how to take your **MCP servers** from prototype to production with **Amazon Bedrock AgentCore**, enabling AI agents to interact with your data infrastructure through natural language, in a secure manner.

### What You'll Build

By the end of this workshop, you'll have deployed a complete MCP infrastructure:

1. **AgentCore Gateway** - Secure entry point to all your MCP Servers
2. **AgentCore Runtime for Athena MCP Server** - Securely hosted MCP Server to interact with Amazon Athena
3. **AgentCore Runtime for S3Vectors MCP Server** - Securely hosted MCP Server to interact with S3Vectors

### Architecture Overview

```
┌─────────────────────┐
│   Any AI Agent      │
│   (Your Frontend)   │
└──────────┬──────────┘
           │ JWT Auth
           ▼
┌─────────────────────┐
│  AgentCore Gateway  │
│   (MCP Protocol)    │
└──────────┬──────────┘
           │ OAuth2
           ▼
┌─────────────────────┐
│  AgentCore Runtime  │
├──────────┬──────────┤
│ Athena   │ S3Vectors│
│   MCP    │   MCP    │
└──────────┴──────────┘
     │          │
     ▼          ▼
  Athena      S3Vectors
```

### Prerequisites

- ✅ AWS Account with appropriate permissions
- ✅ Python 3.11+
- ✅ AWS CLI configured
- ✅ Basic understanding of AWS services (Athena, S3, IAM, Cognito)

### Time Required

⏱️ Approximately 30 minutes

---

## Step 1: Install Dependencies

First, we'll install the required Python packages for AgentCore deployment.

The key package is `bedrock-agentcore-starter-toolkit` which provides the Runtime SDK.

In [ ]:
!python -m venv .venv

In [ ]:
!source .venv/bin/activate

In [ ]:
# Install AgentCore dependencies
!pip install -r requirements.txt --quiet

## Step 2: Import Libraries and Configure Environment

We'll import the necessary libraries and configure our AWS environment.

**Key Components:**
- `bedrock_agentcore_starter_toolkit.Runtime` - SDK for deploying to AgentCore
- `ac_utils` - Helper utilities for IAM roles and Cognito setup
- `boto3` - AWS SDK for Python

In [ ]:
import boto3
import json
import sys
import os
import logging
from pathlib import Path
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import ac_utils as utils

# Configure logging for better visibility
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()]
)
logging.getLogger("strands").setLevel(logging.INFO)

# Set AWS region - modify this if you're using a different region
boto_session = boto3.session.Session()
sts_client = boto3.client("sts")
identity = sts_client.get_caller_identity()
aws_account_id = identity["Account"]
REGION = boto_session.region_name
#Set S3 Location for Athena Query results
os.environ['DEFAULT_S3_OUTPUT_LOCATION'] = f'workshop-data-{aws_account_id}-{REGION}'
# Set Athena Workgroup
os.environ['WORKGROUP'] = 'workshop'
print(f"✅ Environment configured for region: {REGION}")

## Helper Functions

Below are the helper functions we'll use throughout this workshop. These handle:

- **IAM Role Management** - Creating roles with appropriate permissions
- **Cognito Setup** - Configuring authentication pools
- **Gateway Creation** - Setting up the AgentCore Gateway
- **MCP Deployment** - Deploying MCP servers to Runtime
- **Target Configuration** - Connecting MCP servers to the Gateway

You don't need to modify these functions - they're ready to use!

In [ ]:
def create_runtime_execution_role():
    """Create or update IAM role for AgentCore Runtime with Athena and S3Vectors permissions."""
    print("\n🔐 Creating/Updating IAM role for AgentCore Runtime...")
    
    role_name = "agentcore-runtime-mcp-data-role"
    iam_client = boto3.client('iam')
    
    try:
        # Try to get existing role first
        try:
            response = iam_client.get_role(RoleName=role_name)
            role_arn = response['Role']['Arn']
            print(f"⚠️  Role '{role_name}' already exists - updating policies...")
            
            # Delete all existing inline policies
            try:
                policies = iam_client.list_role_policies(RoleName=role_name, MaxItems=100)
                for policy_name in policies.get('PolicyNames', []):
                    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
                    print(f"   Deleted old policy: {policy_name}")
            except Exception as e:
                print(f"   Warning: Could not delete old policies: {e}")
            
            # Recreate with updated permissions
            agentcore_runtime_iam_role = utils.create_agentcore_runtime_role_with_data_permissions("mcp-data")
            role_arn = agentcore_runtime_iam_role['Role']['Arn']
            print(f"✅ Runtime IAM role updated: {role_arn}")
            return role_arn
            
        except iam_client.exceptions.NoSuchEntityException:
            # Role doesn't exist, create it
            print(f"   Role '{role_name}' not found, creating new one...")
            agentcore_runtime_iam_role = utils.create_agentcore_runtime_role_with_data_permissions("mcp-data")
            role_arn = agentcore_runtime_iam_role['Role']['Arn']
            print(f"✅ Runtime IAM role created: {role_arn}")
            return role_arn
    except Exception as e:
        print(f"❌ Error with Runtime IAM role: {e}")
        sys.exit(1)


def create_gateway_iam_role():
    """Create or update IAM role for the Gateway to assume."""
    print("\n🔐 Creating/Updating IAM role for AgentCore Gateway...")
    
    role_name = "ac-gw-mcp-role"
    iam_client = boto3.client('iam')
    
    try:
        # Try to get existing role first
        try:
            response = iam_client.get_role(RoleName=role_name)
            role_arn = response['Role']['Arn']
            print(f"⚠️  Role '{role_name}' already exists - updating policies...")
            
            # Delete all existing inline policies
            try:
                policies = iam_client.list_role_policies(RoleName=role_name, MaxItems=100)
                for policy_name in policies.get('PolicyNames', []):
                    iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
                    print(f"   Deleted old policy: {policy_name}")
            except Exception as e:
                print(f"   Warning: Could not delete old policies: {e}")
            
            # Recreate with updated permissions
            agentcore_gateway_iam_role = utils.create_agentcore_gateway_role(role_name)
            role_arn = agentcore_gateway_iam_role['Role']['Arn']
            print(f"✅ Gateway IAM role updated: {role_arn}")
            return role_arn
            
        except iam_client.exceptions.NoSuchEntityException:
            # Role doesn't exist, create it
            print(f"   Role '{role_name}' not found, creating new one...")
            agentcore_gateway_iam_role = utils.create_agentcore_gateway_role(role_name)
            role_arn = agentcore_gateway_iam_role['Role']['Arn']
            print(f"✅ Gateway IAM role created: {role_arn}")
            return role_arn
    except Exception as e:
        print(f"❌ Error with Gateway IAM role: {e}")
        sys.exit(1)


def create_cognito_pool_for_gateway():
    """Create or get existing Amazon Cognito Pool for inbound authorization to Gateway."""
    print("\n🔑 Creating/Getting Cognito Pool for Gateway (Inbound Auth)...")
    
    USER_POOL_NAME = "sample-agentcore-gateway-pool"
    RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
    RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
    CLIENT_NAME = "sample-agentcore-gateway-client"
    SCOPES = [
        {
            "ScopeName": "invoke",
            "ScopeDescription": "Scope for invoking the agentcore gateway"
        },
    ]
    
    scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
    scope_string = " ".join(scope_names)
    
    cognito = boto3.client("cognito-idp", region_name=REGION)
    
    try:
        # Create or retrieve user pool (utils function already handles this)
        gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
        print(f"   User Pool ID: {gw_user_pool_id}")
        
        # Create or retrieve resource server (utils function already handles this)
        utils.get_or_create_resource_server(
            cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
        )
        print("   Resource server ensured")
        
        # Create or retrieve gateway client. Amazon Quick uses User auth
        # (authorization-code flow) against this inbound client -> oauth_flow="code".
        gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
            cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names,
            oauth_flow="code"
        )
        
        # Get discovery URL
        gw_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration'
        
        print(f"✅ Gateway Cognito Pool ready")
        print(f"   Client ID: {gw_client_id}")
        print(f"   Discovery URL: {gw_cognito_discovery_url}")
        
        return {
            "user_pool_id": gw_user_pool_id,
            "client_id": gw_client_id,
            "client_secret": gw_client_secret,
            "discovery_url": gw_cognito_discovery_url,
            "scope_string": scope_string
        }
    except Exception as e:
        print(f"⚠️  Error with Cognito Pool: {e}")
        print(f"   Continuing with execution...")
        raise


def create_cognito_pool_for_runtime():
    """Create or get existing Amazon Cognito Pool for inbound authorization to Runtime (outbound for Gateway)."""
    print("\n🔑 Creating/Getting Cognito Pool for Runtime (Outbound Auth for Gateway)...")
    
    USER_POOL_NAME = "sample-agentcore-runtime-pool"
    RESOURCE_SERVER_ID = "sample-agentcore-runtime-id"
    RESOURCE_SERVER_NAME = "sample-agentcore-runtime-name"
    CLIENT_NAME = "sample-agentcore-runtime-client"
    SCOPES = [
        {
            "ScopeName": "invoke",
            "ScopeDescription": "Scope for invoking the agentcore runtime"
        },
    ]
    
    scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
    scope_string = " ".join(scope_names)
    
    cognito = boto3.client("cognito-idp", region_name=REGION)
    
    try:
        # Create or retrieve user pool (utils function already handles this)
        runtime_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
        print(f"   User Pool ID: {runtime_user_pool_id}")
        
        # Create or retrieve resource server (utils function already handles this)
        utils.get_or_create_resource_server(
            cognito, runtime_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
        )
        print("   Resource server ensured")
        
        # Create or retrieve M2M client (utils function already handles this)
        runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(
            cognito, runtime_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
        )
        
        # Get discovery URL
        runtime_cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration'
        
        print(f"✅ Runtime Cognito Pool ready")
        print(f"   Client ID: {runtime_client_id}")
        print(f"   Discovery URL: {runtime_cognito_discovery_url}")
        
        return {
            "user_pool_id": runtime_user_pool_id,
            "client_id": runtime_client_id,
            "client_secret": runtime_client_secret,
            "discovery_url": runtime_cognito_discovery_url,
            "scope_string": scope_string
        }
    except Exception as e:
        print(f"⚠️  Error with Runtime Cognito Pool: {e}")
        print(f"   Continuing with execution...")
        raise


def create_agentcore_gateway(gateway_role_arn, gw_cognito_config):
    """Create the AgentCore Gateway or get existing one."""
    print("\n🌐 Creating/Getting AgentCore Gateway...")
    
    gateway_name = 'ac-gateway-mcp-server'
    
    try:
        gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
        
        # Try to list existing gateways to see if one with this name exists
        try:
            list_response = gateway_client.list_gateways()
            existing_gateways = list_response.get('items', [])  # API returns 'items' not 'gateways'
            
            for gateway in existing_gateways:
                if gateway.get('name') == gateway_name:
                    gateway_id = gateway.get('gatewayId')
                    # Get full gateway details to get the URL
                    gateway_details = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
                    gateway_url = gateway_details.get('gatewayUrl')
                    print(f"✅ Using existing Gateway")
                    print(f"   Gateway ID: {gateway_id}")
                    print(f"   Gateway URL: {gateway_url}")
                    return {
                        "gateway_id": gateway_id,
                        "gateway_url": gateway_url
                    }
        except Exception as list_error:
            print(f"   Could not list gateways: {list_error}")
        
        # Gateway doesn't exist, create it
        print(f"   Gateway '{gateway_name}' not found, creating new one...")
        
        auth_config = {
            "customJWTAuthorizer": {
                "allowedClients": [gw_cognito_config["client_id"]],
                "discoveryUrl": gw_cognito_config["discovery_url"]
            }
        }
        
        create_response = gateway_client.create_gateway(
            name=gateway_name,
            roleArn=gateway_role_arn,
            protocolType='MCP',
            protocolConfiguration={
                'mcp': {
                    'supportedVersions': ['2025-03-26'],
                    'searchType': 'SEMANTIC'
                }
            },
            authorizerType='CUSTOM_JWT',
            authorizerConfiguration=auth_config,
            description='AgentCore Gateway with MCP Server targets (Athena + S3Vectors)'
        )
        
        gateway_id = create_response["gatewayId"]
        gateway_url = create_response["gatewayUrl"]
        
        print(f"✅ Gateway created successfully")
        print(f"   Gateway ID: {gateway_id}")
        print(f"   Gateway URL: {gateway_url}")
        
        return {
            "gateway_id": gateway_id,
            "gateway_url": gateway_url
        }
    except Exception as e:
        print(f"❌ Error with Gateway: {e}")
        sys.exit(1)


def deploy_mcp_server_to_runtime(mcp_file, agent_name, runtime_role_arn, runtime_cognito_config, config, env_vars=None):
    """
    Deploy an MCP server to AgentCore Runtime.
    
    Args:
        mcp_file: Name of the MCP server file (e.g., 'athena_mcp.py')
        agent_name: Name for the agent
        runtime_role_arn: ARN of the IAM role for Runtime execution
        runtime_cognito_config: Runtime Cognito configuration
        config: Application configuration
        env_vars: Optional environment variables to set
    
    Returns:
        Dictionary with agent_arn, agent_id, and agent_url
    """
    print(f"\n🚀 Deploying {mcp_file} to AgentCore Runtime...")
    
    # Get the current working directory (notebook directory)
    script_dir = Path.cwd()
    
    # Verify required files exist in the script directory
    required_files = [mcp_file, 'requirements.txt']
    for file in required_files:
        file_path = script_dir / file
        if not file_path.exists():
            raise FileNotFoundError(f"Required file {file} not found at {file_path}")
    print("   All required files found ✓")
    
    # Save current directory and change to script directory
    original_dir = os.getcwd()
    os.chdir(script_dir)
    print(f"   Working directory: {script_dir}")
    
    try:
        # Initialize Runtime
        agentcore_runtime = Runtime()
        
        # Configure auth
        auth_config = {
            "customJWTAuthorizer": {
                "allowedClients": [runtime_cognito_config["client_id"]],
                "discoveryUrl": runtime_cognito_config["discovery_url"]
            }
        }
        
        # Set environment variables if provided
        # if env_vars:
        #     print(f"   Setting environment variables: {list(env_vars.keys())}")
        #     for key, value in env_vars.items():
        #         os.environ[key] = value
        
        # Configure Runtime with custom execution role
        print("   Configuring AgentCore Runtime...")
        print(f"   Using Runtime execution role: {runtime_role_arn}")
        response = agentcore_runtime.configure(
            entrypoint=mcp_file,
            execution_role=runtime_role_arn,  # Use custom role instead of auto-create
            auto_create_ecr=True,
            requirements_file="requirements.txt",
            non_interactive=True,
            region=REGION,
            authorizer_configuration=auth_config,
            protocol="MCP",
            agent_name=agent_name,
            #environment_variables=env_vars or {}
        )
        print("   Configuration completed ✓")
        
        # Launch to Runtime
        print("   Launching MCP server to AgentCore Runtime...")
        print("   This may take several minutes...")
        launch_result = agentcore_runtime.launch(auto_update_on_conflict=True, env_vars=env_vars)
        
        agent_arn = launch_result.agent_arn
        agent_id = launch_result.agent_id
        
        # Construct agent URL
        encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
        agent_url = f'https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT'
        
        print(f"✅ {mcp_file} deployed successfully")
        print(f"   Agent ARN: {agent_arn}")
        print(f"   Agent ID: {agent_id}")
        print(f"   Agent URL: {agent_url}")
        
        return {
            "agent_arn": agent_arn,
            "agent_id": agent_id,
            "agent_url": agent_url
        }
    finally:
        # Always restore the original directory
        os.chdir(original_dir)


def create_oauth_credential_provider(runtime_cognito_config):
    """Create AgentCore Identity OAuth credential provider for outbound auth."""
    print("\n🔐 Creating OAuth credential provider for Gateway outbound auth...")
    
    try:
        identity_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
        
        cognito_provider = identity_client.create_oauth2_credential_provider(
            name="ac-gateway-mcp-server-identity",
            credentialProviderVendor="CustomOauth2",
            oauth2ProviderConfigInput={
                'customOauth2ProviderConfig': {
                    'oauthDiscovery': {
                        'discoveryUrl': runtime_cognito_config["discovery_url"],
                    },
                    'clientId': runtime_cognito_config["client_id"],
                    'clientSecret': runtime_cognito_config["client_secret"]
                }
            }
        )
        
        cognito_provider_arn = cognito_provider['credentialProviderArn']
        print(f"✅ OAuth credential provider created")
        print(f"   Provider ARN: {cognito_provider_arn}")
        
        return cognito_provider_arn
    except Exception as e:
        print(f"❌ Error creating OAuth credential provider: {e}")
        sys.exit(1)


def create_gateway_target(gateway_id, agent_url, credential_provider_arn, runtime_scope_string, target_name):
    """Create a Gateway target for an MCP server or get existing one."""
    print(f"\n🎯 Creating/Getting Gateway target: {target_name}...")
    
    try:
        gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
        
        # Try to list existing targets to see if one with this name exists
        try:
            list_response = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id)
            existing_targets = list_response.get('items', [])  # API returns 'items' not 'targets'
            
            for target in existing_targets:
                if target.get('name') == target_name:
                    target_id = target.get('targetId')
                    print(f"✅ Using existing Gateway target: {target_name}")
                    print(f"   Target ID: {target_id}")
                    return target_id
        except Exception as list_error:
            print(f"   Could not list targets: {list_error}")
        
        # Target doesn't exist, create it
        print(f"   Target '{target_name}' not found, creating new one...")
        
        create_gateway_target_response = gateway_client.create_gateway_target(
            name=target_name,
            gatewayIdentifier=gateway_id,
            targetConfiguration={
                'mcp': {
                    'mcpServer': {
                        'endpoint': agent_url
                    }
                }
            },
            credentialProviderConfigurations=[
                {
                    'credentialProviderType': 'OAUTH',
                    'credentialProvider': {
                        'oauthCredentialProvider': {
                            'providerArn': credential_provider_arn,
                            'scopes': [runtime_scope_string]
                        }
                    }
                },
            ]
        )
        
        target_id = create_gateway_target_response.get('targetId', 'N/A')
        print(f"✅ Gateway target created: {target_name}")
        print(f"   Target ID: {target_id}")
        
        return target_id
    except Exception as e:
        print(f"❌ Error with Gateway target: {e}")
        sys.exit(1)


def verify_gateway_targets(gateway_id):
    """Verify that Gateway targets exist and are READY."""
    print(f"\n✅ Verifying Gateway targets...")
    
    try:
        gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
        list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id)
        
        targets = list_targets_response.get('items', [])  # API returns 'items' not 'targets'
        print(f"   Found {len(targets)} target(s)")
        
        for target in targets:
            target_name = target.get('name', 'Unknown')
            target_status = target.get('status', 'Unknown')
            print(f"   - {target_name}: {target_status}")
        
        return targets
    except Exception as e:
        print(f"⚠️  Warning: Could not verify targets: {e}")
        return []

---

## Step 3: Verify Configuration

Let's verify that the environment variables are properly set:
- Athena workgroup settings
- S3 output locations

These were configured in Step 2 and will be used throughout the deployment.

In [ ]:
# Verify configuration from environment variables
print("✅ Configuration from environment variables:")
print(f"   Athena Workgroup: {os.environ.get('WORKGROUP', 'Not set')}")
print(f"   S3 Output Location: {os.environ.get('DEFAULT_S3_OUTPUT_LOCATION', 'Not set')}")

---

## Step 4: Create IAM Roles

We need two IAM roles:

### 1. Gateway Role (`ac-gw-mcp-role`)
- Allows the Gateway to invoke Runtime endpoints
- Manages request routing

### 2. Runtime Role (`agentcore-runtime-mcp-data-role`)
- Grants access to AWS services (Athena, S3, Glue, Bedrock)
- Used by MCP servers to access data

**Note:** If roles already exist, they'll be updated with the latest permissions.

In [ ]:
# Create Gateway IAM role
gateway_role_arn = create_gateway_iam_role()
print(f"✅ Gateway Role ARN: {gateway_role_arn}")

In [ ]:
# Create Runtime IAM role with data permissions
runtime_role_arn = create_runtime_execution_role()
print(f"✅ Runtime Role ARN: {runtime_role_arn}")

---

## Step 5: Create Cognito User Pools for Authentication

We need two Cognito User Pools for the authentication chain:

### 1. Gateway Pool (Inbound Authentication)
- Client applications authenticate here
- Receives JWT tokens with 'invoke' scope
- Validates incoming requests to the Gateway

### 2. Runtime Pool (Outbound Authentication)
- Gateway authenticates to Runtime using OAuth2
- Enables secure communication between Gateway and Runtime
- Isolates authentication layers

**Security Note:** This two-layer authentication provides defense in depth.

In [ ]:
# Create Gateway Cognito Pool (Inbound Auth)
gw_cognito_config = create_cognito_pool_for_gateway()
print(f"✅ Gateway Cognito Pool created")
print(f"   User Pool ID: {gw_cognito_config['user_pool_id']}")
print(f"   Client ID: {gw_cognito_config['client_id']}")

In [ ]:
# Create Runtime Cognito Pool (Outbound Auth)
runtime_cognito_config = create_cognito_pool_for_runtime()
print(f"✅ Runtime Cognito Pool created")
print(f"   User Pool ID: {runtime_cognito_config['user_pool_id']}")
print(f"   Client ID: {runtime_cognito_config['client_id']}")

---

## Step 6: Create AgentCore Gateway

The Gateway is the entry point for all MCP requests. It:

- **Validates** incoming JWT tokens from clients
- **Routes** requests to appropriate MCP servers
- **Manages** OAuth2 authentication to Runtime
- **Supports** MCP protocol version 2025-03-26

**Protocol:** Model Context Protocol (MCP)  
**Search Type:** Semantic (enables intelligent routing)

In [ ]:
# Create AgentCore Gateway
gateway_info = create_agentcore_gateway(gateway_role_arn, gw_cognito_config)
print(f"✅ Gateway created successfully")
print(f"   Gateway ID: {gateway_info['gateway_id']}")
print(f"   Gateway URL: {gateway_info['gateway_url']}")

---

## Step 7: Deploy Athena MCP Server to Runtime

Now we'll deploy the **Athena MCP Server** which enables:

- 📊 Natural language queries against Athena
- 🗂️ Database and table discovery
- 📈 Query execution and result retrieval

**Deployment Process:**
1. Package the MCP server code
2. Create Docker container
3. Push to ECR (Elastic Container Registry)
4. Deploy to AgentCore Runtime

**⏱️ This step takes 5-10 minutes** as it builds and deploys the container.

In [ ]:
# Prepare Athena environment variables from os.environ
athena_env_vars = {
    "DEFAULT_S3_OUTPUT_LOCATION": os.environ.get("DEFAULT_S3_OUTPUT_LOCATION", ""),
    "WORKGROUP": os.environ.get("WORKGROUP", "")
}

print(f"📋 Athena Configuration:")
print(f"   S3 Output: {athena_env_vars['DEFAULT_S3_OUTPUT_LOCATION']}")
print(f"   Workgroup: {athena_env_vars['WORKGROUP']}")

In [ ]:
# Deploy Athena MCP Server
print("🚀 Deploying Athena MCP Server...")
print("   This may take several minutes...")

athena_agent = deploy_mcp_server_to_runtime(
    mcp_file="athena_mcp.py",
    agent_name="athena_mcp_server",
    runtime_role_arn=runtime_role_arn,
    runtime_cognito_config=runtime_cognito_config,
    config=None,
    env_vars=athena_env_vars
)

print(f"✅ Athena MCP Server deployed")
print(f"   Agent ARN: {athena_agent['agent_arn']}")
print(f"   Agent ID: {athena_agent['agent_id']}")
                                                                                                                                                                       
dockerfile_path = Path(os.getcwd()) / "Dockerfile"
if dockerfile_path.exists():
    dockerfile_path.unlink()

---

## Step 8: Deploy S3Vectors MCP Server to Runtime

Next, we'll deploy the **S3Vectors MCP Server** which provides:

- 🔍 Semantic search over documents
- 📝 Text embedding generation
- 💾 Vector storage in S3
- 🎯 Similarity-based retrieval

**Use Cases:**
- Document Q&A
- Knowledge base search
- Content recommendations

**⏱️ This step also takes 5-10 minutes** for container build and deployment.

In [ ]:
# Deploy S3Vectors MCP Server
print("🚀 Deploying S3Vectors MCP Server...")
print("   This may take several minutes...")

s3vectors_agent = deploy_mcp_server_to_runtime(
    mcp_file="s3vectors_mcp.py",
    agent_name="s3vectors_mcp_server",
    runtime_role_arn=runtime_role_arn,
    runtime_cognito_config=runtime_cognito_config,
    config=None
)

print(f"✅ S3Vectors MCP Server deployed")
print(f"   Agent ARN: {s3vectors_agent['agent_arn']}")
print(f"   Agent ID: {s3vectors_agent['agent_id']}")

---

## Step 9: Create OAuth2 Credential Provider

The OAuth2 Credential Provider enables the Gateway to authenticate with the Runtime.

**How it works:**
1. Gateway needs to call Runtime MCP servers
2. Provider exchanges Gateway credentials for Runtime tokens
3. Gateway includes token in requests to Runtime
4. Runtime validates token and processes request

This creates a secure, automated authentication flow.

In [ ]:
# Create OAuth2 Credential Provider
credential_provider_arn = create_oauth_credential_provider(runtime_cognito_config)
print(f"✅ OAuth2 Credential Provider created")
print(f"   Provider ARN: {credential_provider_arn}")

---

## Step 10: Create Gateway Targets

Gateway Targets connect the Gateway to the deployed MCP servers.

Each target configuration includes:
- **Endpoint URL** - Where the MCP server is hosted
- **Authentication** - OAuth2 credentials for access
- **Protocol** - MCP protocol configuration

We'll create two targets:
1. Athena MCP Target
2. S3Vectors MCP Target

In [ ]:
# Create Athena Gateway Target
athena_target_id = create_gateway_target(
    gateway_id=gateway_info["gateway_id"],
    agent_url=athena_agent["agent_url"],
    credential_provider_arn=credential_provider_arn,
    runtime_scope_string=runtime_cognito_config["scope_string"],
    target_name="athena-mcp-target"
)
print(f"✅ Athena Gateway Target created")
print(f"   Target ID: {athena_target_id}")

In [ ]:
# Create S3Vectors Gateway Target
s3vectors_target_id = create_gateway_target(
    gateway_id=gateway_info["gateway_id"],
    agent_url=s3vectors_agent["agent_url"],
    credential_provider_arn=credential_provider_arn,
    runtime_scope_string=runtime_cognito_config["scope_string"],
    target_name="s3vectors-mcp-target"
)
print(f"✅ S3Vectors Gateway Target created")
print(f"   Target ID: {s3vectors_target_id}")

---

## Step 11: Verify Deployment

Let's verify that all Gateway targets are properly configured and ready.

In [ ]:
# Verify Gateway targets
targets = verify_gateway_targets(gateway_info["gateway_id"])
print(f"\n✅ Verification complete!")
print(f"   Total targets: {len(targets)}")

---

## 🎉 Deployment Complete!

### Summary

You've successfully deployed a complete MCP infrastructure on AWS Bedrock AgentCore!

### What You Built

1. ✅ **AgentCore Gateway** - Secure MCP entry point
2. ✅ **Athena MCP Server** - Data lake query capabilities
3. ✅ **S3Vectors MCP Server** - Semantic search functionality
4. ✅ **Authentication Layer** - Two-tier Cognito security
5. ✅ **IAM Roles** - Proper permission boundaries

## Step 12: Capture Deployment Information

Run the cell below to see your deployment details and save them to a file.

In [ ]:
# Display and save deployment information
print("=" * 70)
print("📋 DEPLOYMENT INFORMATION")
print("=" * 70)

# Calculate token URL
user_pool_id = gw_cognito_config['user_pool_id']
user_pool_id_lowercase = user_pool_id.lower().replace('_', '')
token_url = f"https://{user_pool_id_lowercase}.auth.{REGION}.amazoncognito.com/oauth2/token"

print(f"\n🌐 Gateway Information:")
print(f"   Gateway ID: {gateway_info['gateway_id']}")
print(f"   Gateway URL: {gateway_info['gateway_url']}")

print(f"\n🔑 Authentication:")
print(f"   Token URL: {token_url}")
print(f"   Client ID: {gw_cognito_config['client_id']}")
print(f"   Client Secret: {gw_cognito_config['client_secret']}")

print(f"\n🚀 MCP Servers:")
print(f"   Athena Agent ARN: {athena_agent['agent_arn']}")
print(f"   S3Vectors Agent ARN: {s3vectors_agent['agent_arn']}")

print(f"\n🎯 Gateway Targets:")
print(f"   Athena Target ID: {athena_target_id}")
print(f"   S3Vectors Target ID: {s3vectors_target_id}")

# Save deployment info
deployment_info = {
    "gateway": gateway_info,
    "gateway_role_arn": gateway_role_arn,
    "runtime_role_arn": runtime_role_arn,
    "gateway_auth": {
        "client_id": gw_cognito_config["client_id"],
        "client_secret": gw_cognito_config["client_secret"],
        "discovery_url": gw_cognito_config["discovery_url"],
        "token_url": token_url
    },
    "athena_mcp": athena_agent,
    "s3vectors_mcp": s3vectors_agent,
    "credential_provider_arn": credential_provider_arn,
    "targets": {
        "athena": athena_target_id,
        "s3vectors": s3vectors_target_id
    }
}

deployment_file = Path("deployment_info.json")
with open(deployment_file, 'w') as f:
    json.dump(deployment_info, f, indent=2)

print(f"\n💾 Deployment info saved to: {deployment_file}")
print("=" * 70)

---

## 🚀 Next Steps

### 1. Test Your Deployment

Use the Gateway URL and authentication credentials to test your MCP servers:

```python
# Example: Get an access token
import requests

token_response = requests.post(
    token_url,
    data={
        'grant_type': 'client_credentials',
        'client_id': gw_cognito_config['client_id'],
        'client_secret': gw_cognito_config['client_secret'],
        'scope': gw_cognito_config['scope_string']
    },
    headers={'Content-Type': 'application/x-www-form-urlencoded'}
)

access_token = token_response.json()['access_token']
```

### 2. Integrate with Your Application

Use the Gateway URL in your application to:
- Query Athena databases
- Perform semantic searches
- Build AI-powered data discovery features

### 3. Monitor and Optimize

- Check CloudWatch logs for MCP server activity
- Monitor Athena query performance
- Optimize vector search parameters

### 4. Extend Functionality

Consider adding more MCP servers for:
- Additional data sources
- Custom business logic
- Specialized analytics

---

## 📚 Resources

- [AWS Bedrock AgentCore Documentation](https://aws.amazon.com/bedrock/agentcore/)
- [Model Context Protocol Specification](https://modelcontextprotocol.io/)
- [AgentCore Starter Toolkit](https://github.com/awslabs/amazon-bedrock-agentcore-samples)

---

## 🆘 Troubleshooting

### Common Issues

**Issue:** Deployment takes too long  
**Solution:** Container builds can take 10-15 minutes. Be patient!

**Issue:** Authentication errors  
**Solution:** Verify Cognito pool configuration and client credentials

**Issue:** MCP server not responding  
**Solution:** Check CloudWatch logs for the Runtime execution

**Issue:** Permission denied errors  
**Solution:** Verify IAM roles have correct policies attached

---

## 🎓 What You Learned

- ✅ How to deploy MCP servers to AgentCore Runtime
- ✅ Setting up secure authentication with Cognito
- ✅ Configuring Gateway targets and routing
- ✅ Managing IAM roles for data access
- ✅ Building production-ready AI infrastructure

**Congratulations on deploying your MCP servers securely on AgentCore!** 🎉